In [74]:
# ============================================================
# FIND EXISTING TRAINING DATA VARIABLES
# ============================================================

print("Variables containing 'train':")
print("=" * 60)

for name in dir():
    if "train" in name.lower():
        try:
            obj = globals()[name]
            print(
                f"{name:30s} | "
                f"type = {type(obj).__name__}"
            )

            if hasattr(obj, "shape"):
                print(
                    f"{'':30s} | "
                    f"shape = {obj.shape}"
                )
        except:
            pass

Variables containing 'train':
X_train                        | type = DataFrame
                               | shape = (177576, 21)
X_train_np                     | type = ndarray
                               | shape = (177576, 21)
X_train_processed              | type = DataFrame
                               | shape = (177576, 21)
X_train_tensor                 | type = Tensor
                               | shape = torch.Size([177576, 21])
epoch_train_loss               | type = float
running_train_loss             | type = float
train_dataset                  | type = TensorDataset
train_loader                   | type = DataLoader
train_losses                   | type = list
train_samples                  | type = int
train_test_split               | type = function
y_train                        | type = Series
                               | shape = (177576,)
y_train_processed              | type = ndarray
                               | shape = (177576,)
y_train_tensor 

In [75]:
# ============================================================
# PHASE 6 — LIME EXPLAINER SETUP
# ============================================================

from lime.lime_tabular import LimeTabularExplainer

# Feature names
feature_names = list(X_train_processed.columns)

print("LIME EXPLAINER SETUP")
print("=" * 70)

print(f"Number of features: {len(feature_names)}")

print("\nFeature names:")
for i, feature in enumerate(feature_names):
    print(f"{i:02d} | {feature}")


# ------------------------------------------------------------
# LIME prediction function
# ------------------------------------------------------------

def lime_predict_proba(data):

    """
    Receives NumPy feature arrays from LIME
    and returns probabilities for both classes.

    Column 0 = No Diabetes
    Column 1 = Diabetes
    """

    # Convert NumPy data to PyTorch tensor
    tensor_data = torch.tensor(
        data,
        dtype=torch.float32
    )

    # Evaluation mode for normal DNN prediction
    model.eval()

    # No gradients needed
    with torch.no_grad():

        diabetes_probability = (
            model(tensor_data)
            .squeeze(1)
            .cpu()
            .numpy()
        )

    # Probability of no diabetes
    no_diabetes_probability = (
        1.0 - diabetes_probability
    )

    # LIME expects probability for both classes
    probabilities = np.column_stack([
        no_diabetes_probability,
        diabetes_probability
    ])

    return probabilities


# ------------------------------------------------------------
# Create LIME explainer
# ------------------------------------------------------------

lime_explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=feature_names,
    class_names=[
        "No Diabetes",
        "Diabetes"
    ],
    mode="classification",
    discretize_continuous=False,
    random_state=42
)

print("\n" + "=" * 70)
print("LIME EXPLAINER CREATED SUCCESSFULLY")
print("=" * 70)

print("Training data shape:", X_train_np.shape)
print("Number of features:", len(feature_names))
print("Classes:", ["No Diabetes", "Diabetes"])
print("Mode: classification")
print("Discretize continuous: False")
print("Random state: 42")

LIME EXPLAINER SETUP
Number of features: 21

Feature names:
00 | HighBP
01 | HighChol
02 | CholCheck
03 | BMI
04 | Smoker
05 | Stroke
06 | HeartDiseaseorAttack
07 | PhysActivity
08 | Fruits
09 | Veggies
10 | HvyAlcoholConsump
11 | AnyHealthcare
12 | NoDocbcCost
13 | GenHlth
14 | MentHlth
15 | PhysHlth
16 | DiffWalk
17 | Sex
18 | Age
19 | Education
20 | Income

LIME EXPLAINER CREATED SUCCESSFULLY
Training data shape: (177576, 21)
Number of features: 21
Classes: ['No Diabetes', 'Diabetes']
Mode: classification
Discretize continuous: False
Random state: 42


In [76]:
# ============================================================
# PHASE 6 — STEP 2
# GENERATE FIRST LIME EXPLANATION
# ============================================================

# Select Patient 1 from the test set
patient_index = 0

patient_features = X_test_np[patient_index]

print("GENERATING LIME EXPLANATION")
print("=" * 70)

print(f"Patient index: {patient_index}")
print(f"Number of features: {len(patient_features)}")

# ------------------------------------------------------------
# Generate LIME explanation
# ------------------------------------------------------------

lime_explanation = lime_explainer.explain_instance(
    patient_features,
    lime_predict_proba,
    num_features=21,
    top_labels=1,
    num_samples=5000
)

# ------------------------------------------------------------
# Determine predicted class
# ------------------------------------------------------------

patient_probability = mc_mean[patient_index]

predicted_class = (
    "Diabetes"
    if patient_probability >= 0.5
    else "No Diabetes"
)

# ------------------------------------------------------------
# Display prediction information
# ------------------------------------------------------------

print("\nPATIENT PREDICTION")
print("-" * 70)

print(
    f"MC Mean Probability : "
    f"{patient_probability:.6f}"
)

print(
    f"Predicted Class     : "
    f"{predicted_class}"
)

print(
    f"True Label          : "
    f"{int(test_true_labels[patient_index])}"
)

print(
    f"Predictive Variance : "
    f"{mc_variance[patient_index]:.10f}"
)

print(
    f"Uncertainty Level   : "
    f"{uncertainty_levels[patient_index]}"
)

# ------------------------------------------------------------
# Extract LIME feature contributions
# ------------------------------------------------------------

lime_features = lime_explanation.as_list(
    label=1
)

print("\nLIME FEATURE CONTRIBUTIONS")
print("-" * 70)

for feature, contribution in lime_features:

    direction = (
        "toward Diabetes"
        if contribution > 0
        else "toward No Diabetes"
    )

    print(
        f"{feature:30s} | "
        f"{contribution:+.6f} | "
        f"{direction}"
    )

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("\nLIME VALIDATION")
print("-" * 70)

print(
    "Number of explanation features:",
    len(lime_features)
)

print(
    "Expected maximum features:",
    21
)

print(
    "Explanation generated successfully:",
    len(lime_features) > 0
)

GENERATING LIME EXPLANATION
Patient index: 0
Number of features: 21

PATIENT PREDICTION
----------------------------------------------------------------------
MC Mean Probability : 0.546721
Predicted Class     : Diabetes
True Label          : 1
Predictive Variance : 0.0019651442
Uncertainty Level   : High

LIME FEATURE CONTRIBUTIONS
----------------------------------------------------------------------
GenHlth                        | +0.106410 | toward Diabetes
BMI                            | +0.100567 | toward Diabetes
Age                            | +0.076480 | toward Diabetes
HighBP                         | +0.061613 | toward Diabetes
HighChol                       | +0.054754 | toward Diabetes
CholCheck                      | +0.040783 | toward Diabetes
HeartDiseaseorAttack           | +0.029638 | toward Diabetes
Income                         | -0.029124 | toward No Diabetes
HvyAlcoholConsump              | -0.026854 | toward No Diabetes
Sex                            | +0.020

In [96]:
# ============================================================
# STEP 3A — Prepare and Verify LIME Sample
# ============================================================

# Load the same stratified sample used for SHAP
lime_sample_df = pd.read_csv("stratified_samples.csv")

# Extract the exact 600 test instance IDs
lime_instance_ids = lime_sample_df["test_instance_id"].to_numpy()

# Retrieve the corresponding 21 preprocessed features
X_lime = X_test_processed.iloc[lime_instance_ids].copy()

# Retrieve corresponding true labels
y_lime = y_test_processed[lime_instance_ids]

print("LIME Sample Verification")
print("=" * 60)

print(f"Sample shape:        {lime_sample_df.shape}")
print(f"X_lime shape:        {X_lime.shape}")
print(f"y_lime shape:        {y_lime.shape}")
print(f"Unique instance IDs: {len(np.unique(lime_instance_ids))}")

# Verify exact same instances as SHAP
same_instances = np.array_equal(
    shap_instance_ids,
    lime_instance_ids
)

print(f"\nSame instances as SHAP: {same_instances}")

# Stratum distribution
print("\nStratum counts:")
print(
    lime_sample_df["uncertainty_stratum"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"])
)

# Check feature count
print(f"\nFeature count: {X_lime.shape[1]}")

# Check for invalid values
print(f"Contains NaN: {X_lime.isna().any().any()}")
print(f"Contains Inf: {np.isinf(X_lime.to_numpy()).any()}")

print("\nFirst 5 instance IDs:")
print(lime_instance_ids[:5])

LIME Sample Verification
Sample shape:        (600, 3)
X_lime shape:        (600, 21)
y_lime shape:        (600,)
Unique instance IDs: 600

Same instances as SHAP: True

Stratum counts:
uncertainty_stratum
LOW       200
MEDIUM    200
HIGH      200
Name: count, dtype: int64

Feature count: 21
Contains NaN: False
Contains Inf: False

First 5 instance IDs:
[ 529  850  905 1029 1041]


In [99]:
# ============================================================
# STEP 3B — Final LIME Explainer Verification
# ============================================================

print("LIME Explainer Verification")
print("=" * 60)

print(f"Explainer type: {type(lime_explainer).__name__}")

print(f"Number of feature names: "
      f"{len(lime_explainer.feature_names)}")

print(f"Feature names match X_lime: "
      f"{lime_explainer.feature_names == X_lime.columns.tolist()}")

print(f"Mode: {lime_explainer.mode}")

# ------------------------------------------------------------
# Verify LIME prediction function
# ------------------------------------------------------------

test_input = X_lime.iloc[[0]].to_numpy(dtype=np.float32)

test_prediction = lime_predict_proba(test_input)

print("\nPrediction Function Check")
print("-" * 60)
print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_prediction.shape}")
print(f"Output:       {test_prediction}")

# ------------------------------------------------------------
# Probability checks
# ------------------------------------------------------------

probability_sum = test_prediction.sum(axis=1)

print(f"\nProbabilities sum to 1: "
      f"{np.allclose(probability_sum, 1.0)}")

print(f"Contains NaN: "
      f"{np.isnan(test_prediction).any()}")

print(f"Contains Inf: "
      f"{np.isinf(test_prediction).any()}")

# ------------------------------------------------------------
# Check probability values are valid
# ------------------------------------------------------------

print(
    f"All probabilities between 0 and 1: "
    f"{np.all((test_prediction >= 0) & (test_prediction <= 1))}"
)

LIME Explainer Verification
Explainer type: LimeTabularExplainer
Number of feature names: 21
Feature names match X_lime: True
Mode: classification

Prediction Function Check
------------------------------------------------------------
Input shape:  (1, 21)
Output shape: (1, 2)
Output:       [[0.4981094 0.5018906]]

Probabilities sum to 1: True
Contains NaN: False
Contains Inf: False
All probabilities between 0 and 1: True


In [100]:
# ============================================================
# STEP 3C — Single-Instance LIME Validation
# ============================================================

# Select the first sampled instance
lime_test_instance = X_lime.iloc[0].to_numpy(dtype=np.float32)

# Generate LIME explanation
lime_single_explanation = lime_explainer.explain_instance(
    lime_test_instance,
    lime_predict_proba,
    num_features=21
)

# Extract feature contributions
lime_single_contributions = lime_single_explanation.as_list()

print("Single-Instance LIME Validation")
print("=" * 60)

print(f"Instance ID:       {lime_instance_ids[0]}")
print(f"True label:        {y_lime[0]:.0f}")

# Model probability
lime_prediction = lime_predict_proba(
    lime_test_instance.reshape(1, -1)
)[0]

print(f"No Diabetes probability: {lime_prediction[0]:.6f}")
print(f"Diabetes probability:    {lime_prediction[1]:.6f}")

print(f"\nNumber of LIME features explained: "
      f"{len(lime_single_contributions)}")

print("\nFeature Contributions")
print("-" * 60)

for feature, contribution in lime_single_contributions:
    print(f"{feature:<30} {contribution:+.8f}")

# ------------------------------------------------------------
# Explanation object verification
# ------------------------------------------------------------

print("\nLIME Explanation Object")
print("-" * 60)
print(f"Explanation type: "
      f"{type(lime_single_explanation).__name__}")

print(
    f"Available labels: "
    f"{lime_single_explanation.available_labels()}"
)

Single-Instance LIME Validation
Instance ID:       529
True label:        0
No Diabetes probability: 0.498109
Diabetes probability:    0.501891

Number of LIME features explained: 21

Feature Contributions
------------------------------------------------------------
GenHlth                        +0.10078896
BMI                            +0.10066681
Age                            +0.07506873
HighBP                         +0.06135176
HighChol                       +0.04978108
CholCheck                      +0.03721505
HeartDiseaseorAttack           +0.02917695
Income                         -0.02780905
HvyAlcoholConsump              -0.02750253
MentHlth                       -0.02713350
Sex                            +0.01945961
DiffWalk                       +0.01363036
Stroke                         +0.01232809
Education                      -0.01231915
AnyHealthcare                  +0.00575146
PhysActivity                   -0.00387397
NoDocbcCost                    +0.00335387
Fr

In [104]:
# ============================================================
# STEP 3D — LIME on All 600 Sampled Instances
# ============================================================

import pickle
import time

print("Starting LIME computation for 600 instances...")
print("=" * 60)

lime_outputs = []

start_time = time.time()

for i in range(len(X_lime)):

    # Current instance
    instance_id = int(lime_instance_ids[i])
    instance = X_lime.iloc[i].to_numpy(dtype=np.float32)

    # Generate LIME explanation
    explanation = lime_explainer.explain_instance(
        instance,
        lime_predict_proba,
        num_features=21
    )

    # Get model probabilities
    probabilities = lime_predict_proba(
        instance.reshape(1, -1)
    )[0]

    # Store all information needed later
    output = {
        "test_instance_id": instance_id,
        "sigma_squared": float(
            lime_sample_df.iloc[i]["sigma_squared"]
        ),
        "uncertainty_stratum": lime_sample_df.iloc[i][
            "uncertainty_stratum"
        ],
        "true_label": int(y_lime[i]),
        "no_diabetes_probability": float(probabilities[0]),
        "diabetes_probability": float(probabilities[1]),
        "lime_explanation": explanation.as_list(),
        "lime_score": explanation.score
    }

    lime_outputs.append(output)

    # Progress update
    if (i + 1) % 50 == 0:
        elapsed = time.time() - start_time
        print(
            f"Completed {i + 1}/600 instances "
            f"({elapsed:.1f} seconds)"
        )

# ------------------------------------------------------------
# Save LIME outputs
# ------------------------------------------------------------

with open("lime_outputs.pkl", "wb") as f:
    pickle.dump(lime_outputs, f)

elapsed_total = time.time() - start_time

print("\nLIME Computation Completed")
print("=" * 60)

print(f"Total instances explained: {len(lime_outputs)}")
print(f"Total computation time:     {elapsed_total:.1f} seconds")

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\nVerification")
print("-" * 60)

print(
    f"Exactly 600 explanations: "
    f"{len(lime_outputs) == 600}"
)

print(
    f"Each explanation has 21 features: "
    f"{all(len(x['lime_explanation']) == 21 for x in lime_outputs)}"
)

print(
    f"Unique instance IDs: "
    f"{len(set(x['test_instance_id'] for x in lime_outputs))}"
)

print("\nStratum counts:")

lime_output_strata = pd.Series(
    [x["uncertainty_stratum"] for x in lime_outputs]
).value_counts().reindex(
    ["LOW", "MEDIUM", "HIGH"],
    fill_value=0
)

print(lime_output_strata)

print("\nSaved file:")
print("lime_outputs.pkl")

Starting LIME computation for 600 instances...
Completed 50/600 instances (0.8 seconds)
Completed 100/600 instances (1.4 seconds)
Completed 150/600 instances (2.1 seconds)
Completed 200/600 instances (2.7 seconds)
Completed 250/600 instances (3.4 seconds)
Completed 300/600 instances (4.0 seconds)
Completed 350/600 instances (4.6 seconds)
Completed 400/600 instances (5.3 seconds)
Completed 450/600 instances (6.1 seconds)
Completed 500/600 instances (6.7 seconds)
Completed 550/600 instances (7.4 seconds)
Completed 600/600 instances (8.1 seconds)

LIME Computation Completed
Total instances explained: 600
Total computation time:     8.1 seconds

Verification
------------------------------------------------------------
Exactly 600 explanations: True
Each explanation has 21 features: True
Unique instance IDs: 600

Stratum counts:
LOW       200
MEDIUM    200
HIGH      200
Name: count, dtype: int64

Saved file:
lime_outputs.pkl
